# Recoverability — coverage–auditability tension (Part A, then **STOP**)

Decisive cheap test on the SAME frozen representation the conformity scores live on, per
(backbone × dataset). **Recoverability**: can a standardized probe recover the spurious attribute
(place / Male) from the features? (test AUROC, in-domain train→test, ≥3 seeds + 95% CI).
**Worst-group coverage**: best achievable via Mondrian (group-conditional) APS @ρ=0.95.

Verdict per cell: `tension_dead` (good coverage + AUROC≥0.65) / `tension_alive` (good coverage only
at AUROC≤0.55) / `ambiguous`.

**STOP after Part A** for human review. **Part B** (project out top-k spurious directions, trace the
coverage↔recoverability tradeoff) runs ONLY if some cell is `ambiguous`/`tension_alive`. Reuses the
cached features (no re-extraction if the grid already ran). GPU runtime recommended (cache-miss only).

## 0. Parameters — **EDIT THESE**

In [ ]:
REPO_SOURCE   = "git"
REPO_URL      = "https://github.com/octadion/vgscp.git"
REPO_BRANCH   = "main"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"

SEEDS         = 3
N_SPLITS      = 10
CELEBA_RESNET_MAX_TRAIN = 30000
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE  = "kaggle"   # "kaggle" (needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE   = ""
import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU + install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib torchvision", shell=True)

## 2. Mount Drive + repo + datasets (reuses cached features)

In [ ]:
from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT; print("CelebA root:", CELEBA_ROOT)
else: print(f"[note] CelebA unavailable (source={CELEBA_SOURCE}) -> Waterbirds-only.")
for c in ("cache_clip", "cache_resnet"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True); os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")

## 3. Build GridData per (backbone × dataset) — cache hit if the grid already ran

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224, "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN
    return base

KEYS = [("waterbirds", "resnet50_erm"), ("waterbirds", "clip_vitb32")]
if CELEBA_OK: KEYS += [("celeba", "resnet50_erm"), ("celeba", "clip_vitb32")]
data, skipped = {}, []
for ds, bb in KEYS:
    try:
        t = time.time(); data[(bb, ds)] = build_griddata(ds, bb, cfg_for(ds), seed=0)
        print(f"[built] {bb}/{ds} ({(time.time()-t)/60:.1f} min)")
    except Exception as e:
        skipped.append((bb, ds)); print(f"[SKIP] {bb}/{ds}: {e}")
print("cells:", list(data.keys()), "| skipped:", skipped)

## 4. Part A — recoverability AUROC + worst-group Mondrian coverage + verdict → STOP

In [ ]:
from study_robust_train.recoverability import run_part_a, write_recoverability_md, needs_part_b

A = run_part_a(data, seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)
write_recoverability_md(A, "RECOVERABILITY.md")   # Part A only (the STOP deliverable)
for key, c in A["cells"].items():
    r, cov = c["recoverability"], c["coverage"]
    print(f"{key}: recoverability AUROC={r['auroc_mean']:.3f} CI[{r['ci'][0]:.3f},{r['ci'][1]:.3f}] | "
          f"worst-group cov={cov['worst_group_cov_mean']:.3f} | verdict={c['verdict']}")
print("\nPart B needed (any ambiguous/tension_alive)?:", needs_part_b(A))
from IPython.display import Markdown, display
display(Markdown(open("RECOVERABILITY.md", encoding="utf-8").read()))

## 5. STOP — Part A complete
Hand the per-cell table + verdict to the researcher. Run Part B below **only if** the line above says
Part B is needed (some cell `ambiguous`/`tension_alive`). If every cell is `tension_dead`, the tension
is dead and Part B is unnecessary.

## 6. Part B (gated) — project out top-k spurious directions; trace coverage↔recoverability

In [ ]:
from study_robust_train.recoverability import run_part_b, make_part_b_figure, write_recoverability_md, needs_part_b

if needs_part_b(A):
    B = run_part_b(data, ks=(0, 1, 5, 20), seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)
    os.makedirs("results/study", exist_ok=True)
    fig = make_part_b_figure(B, "results/study/recoverability_partB.png")
    write_recoverability_md(A, "RECOVERABILITY.md", part_b=B, fig_path="results/study/recoverability_partB.png")
    from IPython.display import Image, Markdown, display
    display(Markdown(open("RECOVERABILITY.md", encoding="utf-8").read())); display(Image(fig))
else:
    print("All cells tension_dead -> Part B NOT needed (per spec). Tension is DEAD; STOP.")